In [ ]:
import requests
import pandas as pd
from datetime import datetime

print("[1/4] กำลังดึงข้อมูลจาก Open-Meteo API (ข้อมูล 15 ปี อาจใช้เวลาสักครู่)")

# 1. ตั้งค่าพารามิเตอร์สำหรับการดึงข้อมูล
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 16.3354444,
    "longitude": 102.8266944,
    "start_date": "2003-01-01",
    "end_date": "2024-12-31",
    "hourly": "precipitation,temperature_2m,dew_point_2m,apparent_temperature,relative_humidity_2m,vapour_pressure_deficit,wind_speed_10m,wind_direction_10m,wind_gusts_10m,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_high,soil_temperature_0_to_7cm,soil_moisture_0_to_7cm",
    "timezone": "Asia/Bangkok"
}

# ยิง Request เพื่อขอข้อมูลดิบ
response = requests.get(url, params=params)
data = response.json()

print("ดึงข้อมูลสำเร็จ! กำลังแปลงข้อมูลลงตาราง...")

# 2. แปลงข้อมูล JSON เป็นตาราง
df_hourly = pd.DataFrame(data['hourly'])
df_hourly['time'] = pd.to_datetime(df_hourly['time'])
df_hourly.set_index('time', inplace=True)

print("⏳ [2/4] กำลังยุบรวมข้อมูลจาก รายชั่วโมง -> รายวัน...")

# 3. กำหนดเงื่อนไขการคำนวณรายวัน
daily_logic = {
    'precipitation': ['sum'],
    'temperature_2m': ['mean', 'max', 'min'],
    'apparent_temperature': ['mean', 'max', 'min'],
    'dew_point_2m': ['mean', 'max', 'min'],
    'relative_humidity_2m': ['mean', 'max', 'min'],
    'wind_speed_10m': ['mean', 'max'],
    'wind_gusts_10m': ['max'],
    'wind_direction_10m': ['mean'],
    'cloud_cover': ['mean'],
    'cloud_cover_low': ['mean'],
    'cloud_cover_mid': ['mean'],
    'cloud_cover_high': ['mean'],
    'soil_temperature_0_to_7cm': ['mean'],
    'soil_moisture_0_to_7cm': ['mean'],
    'vapour_pressure_deficit': ['mean']
}

# สั่งคำนวณเป็นรายวัน
df_daily = df_hourly.resample('D').agg(daily_logic)

# แปลงชื่อคอลัมน์
df_daily.columns = [f"{col[0]}_{col[1]}" for col in df_daily.columns]

print("[3/4] กำลังยุบรวมข้อมูลจาก รายวัน -> รายเดือน...")

# 4. กำหนดเงื่อนไขการคำนวณรายเดือน
monthly_logic = {}
for col in df_daily.columns:
    if 'precipitation_sum' in col:
        monthly_logic[col] = 'sum'
    else:
        monthly_logic[col] = 'mean'

# สั่งคำนวณเป็นรายเดือน (ไม่มีการปัดเศษแล้ว)
df_monthly = df_daily.resample('ME').agg(monthly_logic)

print("⏳ [4/4] กำลังบันทึกไฟล์ (Exporting)...")

# 5. เซฟออกมาเป็นไฟล์ CSV และ Excel
output_csv = r"D:\DF\rain_data\test_weather_monthly_summary.csv"
output_excel = r"D:\DF\rain_data\test_weather_monthly_summary.xlsx"

df_monthly.to_csv(output_csv)
df_monthly.to_excel(output_excel)

print(f"เสร็จเรียบร้อย! ข้อมูลถูกบันทึกแบบทศนิยมเต็มๆ ไว้ที่ไฟล์:")
print(f" {output_csv}")
print(f" {output_excel}")

ModuleNotFoundError: No module named 'pandas'

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 2.6 MB/s eta 0:00:04
   ----- ---------------------------------- 1.3/9.7 MB 2.4 MB/s eta 0:00:04
   ------- -------------------------------- 1.8/9.7 MB 2.5 MB/s eta 0:00:04
   -------- ------------------------------- 2.1/9.7 MB 2.3 MB/s eta 0:00:04
   --------- ------------------------------ 2.4/9.7 MB 1.9 MB/s eta 0:00:04
   --------- ------------------------------ 2.4/9.7 MB 1.9 MB/s eta 0:00:04
   ----------- ---------------------------- 2.9/9.7 MB 1.7 MB/s eta 0:00:04
   ------------ --------------------------- 3.1/9.7 MB 1.7 MB/s eta 0:00:04
   --------------- ------------------------ 3.7/9.7 MB 1.8 MB/s eta 0:00:04
   ----------------- ---------------------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


พร้อมใช้งานแล้ว
